In [4]:
import pandas as pd

# ---------------------------
# Load CSVs
# ---------------------------
billings = pd.read_csv('../../dataset/02_basic_clean/billings.csv')
calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')

# ---------------------------
# Convert to datetime
# ---------------------------
billings['Prospect_Renewal_Date'] = pd.to_datetime(billings['Prospect_Renewal_Date'])
calls['Call_Date'] = pd.to_datetime(calls['Call_Date'])

# ---------------------------
# Find Co_Ref with multiple calls (BEFORE MERGE)
# ---------------------------
multi_call_counts = calls['Co_Ref'].value_counts()
multi_call_ids = multi_call_counts[multi_call_counts > 1].index

print("🔁 Co_Ref with multiple calls:")
print(multi_call_counts[multi_call_counts > 1])

# Optional: view full rows of those customers
multi_call_rows = calls[calls['Co_Ref'].isin(multi_call_ids)]
print("\n📌 Full rows for Co_Ref with multiple calls:")
print(multi_call_rows.sort_values('Co_Ref'))

# ---------------------------
# Merge datasets
# ---------------------------
merged = pd.merge(billings, calls, on='Co_Ref', how='inner')

# ---------------------------
# 14-day window filter
# Call must be BEFORE renewal date and within 14 days
# ---------------------------
merged['days_diff'] = (merged['Prospect_Renewal_Date'] - merged['Call_Date']).dt.days

result = merged[(merged['days_diff'] >= 0) & (merged['days_diff'] <= 14)]

# ---------------------------
# Add call count per Co_Ref (AFTER FILTER)
# ---------------------------
result['call_count_per_CoRef'] = result.groupby('Co_Ref')['Co_Ref'].transform('count')

# ---------------------------
# Show Co_Ref with multiple rows in FINAL result
# ---------------------------
print("\n📊 Co_Ref with multiple rows in final dataset:")
print(result['Co_Ref'].value_counts()[result['Co_Ref'].value_counts() > 1])

# ---------------------------
# Save final output
# ---------------------------
result = result.drop(columns=['days_diff'])
result.to_csv('../../dataset/02_basic_clean/final_merged.csv', index=False)

print("\n✅ Done! Saved as final_merged.csv")

C:\Users\Asus\AppData\Local\Temp\ipykernel_14964\1154364657.py:6: DtypeWarning: Columns (10,11,14,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  billings = pd.read_csv('../../dataset/02_basic_clean/billings.csv')
C:\Users\Asus\AppData\Local\Temp\ipykernel_14964\1154364657.py:7: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,24,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
C:\Users\Asus\AppData\Local\Temp\ipykernel_14964\1154364657.py:13: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  calls['Call_Date'] = pd.to_datetime(calls['Call_Date'])


🔁 Co_Ref with multiple calls:
Co_Ref
AD1657    161
UW3387    108
ZF9958     77
XG1277     77
YT5805     67
         ... 
BL4389      2
AZ0370      2
QA5628      2
AD8718      2
JV1670      2
Name: count, Length: 27717, dtype: int64

📌 Full rows for Co_Ref with multiple calls:
             Call_ID Call_Direction  Co_Ref  Call_Date Churn_Category  \
140717  5.070000e+11       Outbound  AA0584 2024-05-01            NaN   
61805   5.070000e+11       Outbound  AA0584 2024-05-01            NaN   
83948   5.900000e+11       Outbound  AA0641 2024-11-11            NaN   
81202   5.900000e+11       Outbound  AA0641 2024-11-11            NaN   
128766  5.900000e+11       Outbound  AA0641 2024-11-11            NaN   
...              ...            ...     ...        ...            ...   
68571   6.780000e+11      OUT_BOUND  ok0108 2025-07-14            NaN   
135060  5.920000e+11        Inbound  ok0108 2024-12-05            NaN   
13481   5.110000e+11        Inbound  ok0108 2024-07-04            

C:\Users\Asus\AppData\Local\Temp\ipykernel_14964\1154364657.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result['call_count_per_CoRef'] = result.groupby('Co_Ref')['Co_Ref'].transform('count')



📊 Co_Ref with multiple rows in final dataset:
Co_Ref
LO2530    18
AM7128    16
DK1892    15
AW0101    14
TE2964    14
          ..
YQ1081     2
XW0273     2
UL3591     2
LY0196     2
WZ4380     2
Name: count, Length: 12515, dtype: int64

✅ Done! Saved as final_merged.csv
